In [1]:
import sys
import numpy as np

from Rain.Rain import Rain

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [3]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # network parameters
        hidden_units = 256
        dropout = 0.45
        input_size = 784
        num_labels = 10
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_units, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

In [4]:
model = Model()
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "pytorch",
      "params": {
        "loss": nn.CrossEntropyLoss(),
        "optimizer": optim.Adam(model.parameters(), lr=0.001)

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [5]:
X_train, y_train = get_train_data()
y_train = np.argmax(y_train, axis=1)

In [6]:
rain = Rain(config, model)

2023-07-07 15:26:23,190 [DEBUG] [Rain] Rain is initialized
2023-07-07 15:26:23,192 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 15:26:23,193 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 15:26:23,194 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 15:26:23,194 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 15:26:23,196 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 15:26:23,197 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 15:26:23,199 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [7]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 15:26:23,210 [INFO] [Provisioner] provisioner is serving
2023-07-07 15:26:23,211 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 15:26:23,213 [INFO] [Coordinator] coordinator is serving
2023-07-07 15:26:23,214 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 15:26:23,220 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 15:26:23,221 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 15:26:23,223 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 15:26:23,225 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 15:26:23,226 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50152/
2023-07-07 15:26:23,228 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-07 15:26:23,229 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50153/
2023-07-07 15:26

Epoch [1/5], Loss: 0.1900, Accuracy: 0.9446
Epoch [1/5], Loss: 0.1871, Accuracy: 0.9435
Epoch [2/5], Loss: 0.1617, Accuracy: 0.9520
Epoch [2/5], Loss: 0.1544, Accuracy: 0.9539
Epoch [3/5], Loss: 0.1376, Accuracy: 0.9582
Epoch [3/5], Loss: 0.1375, Accuracy: 0.9587
Epoch [4/5], Loss: 0.1179, Accuracy: 0.9647
Epoch [4/5], Loss: 0.1195, Accuracy: 0.9637


2023-07-07 15:26:39,201 [DEBUG] [DividerAmbassador] divider received: 'Executed!' after executing the model on worker2
2023-07-07 15:26:39,202 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-07 15:26:39,203 [DEBUG] [DividerAmbassador] divider received: 'Executed!' after executing the model on worker3
2023-07-07 15:26:39,205 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_2_trained.pkl from worker3
2023-07-07 15:26:39,259 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-07 15:26:39,262 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_2_trained.pkl from worker3 successfully
2023-07-07 15:26:39,265 [DEBUG] [DeepLearning] Error in reducing the gradients: 'NoneType' object is not iterable
2023-07-07 15:26:39,265 [DEBUG] [DeepLearning] Iteration 2/3 complete.
2023-07-07 15:26:39,266 [DEBUG] [DeepLearni

Epoch [5/5], Loss: 0.1093, Accuracy: 0.9673
Epoch [5/5], Loss: 0.1037, Accuracy: 0.9678
sending data to divider
sending data to divider


2023-07-07 15:26:39,434 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-07 15:26:39,435 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-07 15:26:39,436 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker1
2023-07-07 15:26:39,437 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker2
2023-07-07 15:26:39,437 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 3
2023-07-07 15:26:39,439 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker3
2023-07-07 15:26:39,454 [DEBUG] [DividerAmbassador] divider received: 'Executed!' after executing the model on worker1
2023-07-07 15:26:39,457 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-07 15:26:39,464 [DEBUG] [DividerA

In [8]:
def evaluate_model(model, X_test, y_test, batch_size):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()

    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0
    
    for i, (data, labels) in enumerate(test_loader):
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [ ]:
X_test, y_test = get_test_data()
y_test = np.argmax(y_test, axis=1)
acc = evaluate_model(model, X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 94.7%


2023-07-07 15:27:07,633 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 15:27:07,827 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-07 15:27:07,827 [DEBUG] [Provisioner] Received '' from the coordinator to send status
2023-07-07 15:27:07,828 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
2023-07-07 15:27:07,829 [DEBUG] [Provisioner] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
2023-07-07 15:27:10,303 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-07 15:27:10.561746: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, r